# FineWeb-Edu `sample-10BT` → fixed-length corpus (EDA)

Self-contained dev notebook. Streams `HuggingFaceFW/fineweb-edu` (`sample-10BT`), inspects schema and examples, tokenizes with GPT-2, and packs EOS-delimited documents into exact `sample_length`-token windows.

All sizes are driven by `CorpusRegressionConfig` so the notebook stays in sync with the rest of the experiment.

In [ ]:
from src.config.base import BaseConfig


class CorpusRegressionConfig(BaseConfig):
    sample_length: int  # canonical 128
    num_samples: int  # canonical 50_000

    num_regression_bins: int  # Don't worry about this for now. Instantiate to 8


cfg = CorpusRegressionConfig(sample_length=128, num_samples=50_000, num_regression_bins=8)
cfg

In [ ]:
from __future__ import annotations

import itertools

import plotly.express as px
import polars as pl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from src import get_repo_base

dataset = "HuggingFaceFW/fineweb-edu"
dataset_config = "sample-10BT"
tokenizer_name = "gpt2"
eda_sample = 5_000  # rows pulled for distribution plots; separate from the packed corpus
seed = 42

out_path = (
    get_repo_base()
    / "artifacts"
    / "corpus-regression"
    / f"fineweb_edu_{cfg.num_samples}_x_{cfg.sample_length}.parquet"
)
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path

## 1. Stream the dataset

`streaming=True` avoids the full ~28 GB download. Shuffle with a buffer before taking anything so the EDA sample isn't just the first shard.

In [ ]:
ds_stream = load_dataset(dataset, name=dataset_config, split="train", streaming=True)
ds_stream = ds_stream.shuffle(seed=seed, buffer_size=10_000)
ds_stream

In [ ]:
first = next(iter(ds_stream))
print("fields:", list(first.keys()))
for k, v in first.items():
    s = repr(v)
    print(f"  {k:>14}: {s[:120]}{' ...' if len(s) > 120 else ''}")

## 2. Pull an EDA sample

Take `EDA_SAMPLE` rows into memory for quick polars/plotly analysis. This is separate from the final packed corpus.

In [ ]:
rows = list(itertools.islice(ds_stream, eda_sample))
eda = pl.DataFrame({
    "text": [r["text"] for r in rows],
    "token_count": [r["token_count"] for r in rows],
    "language": [r.get("language") for r in rows],
    "score": [r.get("score") for r in rows],
    "url": [r.get("url") for r in rows],
})
eda = eda.with_columns(char_len=pl.col("text").str.len_chars())
eda.select("token_count", "char_len", "score", "language").describe()

In [ ]:
fig = px.histogram(
    {"token_count": eda["token_count"].to_list()},
    x="token_count",
    nbins=80,
    log_x=True,
    title=f"FineWeb-Edu {dataset_config}: token_count ({eda_sample} docs)",
)
fig.add_vline(x=cfg.sample_length, line_dash="dash", annotation_text=f"sample_length={cfg.sample_length}")
fig.show()

In [ ]:
q = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.99]
tc = eda["token_count"]
print("token_count quantiles:")
for qi in q:
    print(f"  p{int(qi * 100):>2}: {tc.quantile(qi):>8.0f}")
short = (tc < cfg.sample_length).sum() / len(tc)
print(f"\n{short:.1%} of docs are shorter than {cfg.sample_length} tokens → packing matters.")

## 3. Eyeball raw examples

One short doc, one medium, one long — so you can see the range of content that will get packed.

In [ ]:
picks = (
    eda
    .sort("token_count")
    .with_row_index()
    .filter(
        pl.col("index").is_in([
            len(eda) // 20,
            len(eda) // 2,
            len(eda) - len(eda) // 20,
        ])
    )
)
for row in picks.iter_rows(named=True):
    print(f"--- token_count={row['token_count']}  score={row['score']} ---")
    print(row["text"][:600].rstrip() + (" …" if len(row["text"]) > 600 else ""))
    print()

## 4. Tokenize one doc

`token_count` in the dataset was computed with GPT-2's tokenizer, so our counts should match closely.

In [ ]:
tok = AutoTokenizer.from_pretrained(tokenizer_name)
assert tok.eos_token_id is not None
eos = tok.eos_token_id

sample_text = rows[0]["text"]
ids = tok.encode(sample_text, add_special_tokens=False)
print(f"reported token_count: {rows[0]['token_count']}")
print(f"our len(ids):         {len(ids)}")
print(f"first 32 tokens:      {ids[:32]}")
print(f"decoded first 32:     {tok.decode(ids[:32])!r}")

## 5. Pack into fixed-length windows

EOS-delimited packing: tokenize each doc, append EOS as a boundary marker, extend a rolling buffer, emit fixed `cfg.sample_length` chunks until we've collected `cfg.num_samples`. Re-streams from scratch so the EDA sample above doesn't bias the final corpus.

In [ ]:
ds_pack = load_dataset(dataset, name=dataset_config, split="train", streaming=True).shuffle(
    seed=seed, buffer_size=10_000
)

chunks: list[list[int]] = []
buf: list[int] = []
docs_seen = 0
bar = tqdm(total=cfg.num_samples, desc=f"packing {cfg.sample_length}-token windows")

for ex in ds_pack:
    text = ex["text"].strip()
    if not text:
        continue
    docs_seen += 1
    ids = tok.encode(text, add_special_tokens=False)
    ids.append(eos)
    buf.extend(ids)
    while len(buf) >= cfg.sample_length and len(chunks) < cfg.num_samples:
        chunks.append(buf[: cfg.sample_length])
        del buf[: cfg.sample_length]
        bar.update(1)
    if len(chunks) >= cfg.num_samples:
        break
bar.close()
print(f"emitted {len(chunks)} chunks from {docs_seen} documents ({docs_seen / len(chunks):.2f} docs/chunk)")

## 6. Inspect packed windows

Decode a few back to text and count how many document boundaries (EOS) sit inside each window.

In [ ]:
eos_per_chunk = pl.Series("eos_per_chunk", [c.count(eos) for c in chunks])
print(eos_per_chunk.describe())

px.histogram(
    {"eos_per_chunk": eos_per_chunk.to_list()},
    x="eos_per_chunk",
    nbins=int(eos_per_chunk.max()) + 1,
    title=f"Document boundaries per {cfg.sample_length}-token window",
).show()

In [ ]:
for i in (0, 1, 2, len(chunks) // 2, len(chunks) - 1):
    ids = chunks[i]
    boundaries = ids.count(eos)
    decoded = tok.decode(ids).replace(tok.eos_token, " █ ")
    print(f"--- chunk {i}  boundaries={boundaries} ---")
    print(decoded)
    print()

## 7. Persist

Parquet under `artifacts/corpus-regression/`. One `input_ids` column per row, `len == cfg.sample_length`.

In [ ]:
corpus = pl.DataFrame({"input_ids": chunks})
print(corpus.schema)
print(corpus.head(2))

corpus.write_parquet(out_path)
print(f"wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")